# RBF with Ellipsoid Constraint — Implicit Fitting Using Radial Basis Functions
## Reproducible Workflow — Li et al. (2004)

This notebook demonstrates the Python implementation of the ellipsoid-specific least-squares fitting
algorithm described in:

> Li, Q., et al., J. G. (2004).  
> *Radial basis functions for surface reconstruction from unorganised point clouds with applications to bone reconstruction.*  
> *Computer Graphics Forum*, 23(1), 67–78. Wiley-Blackwell.  
> https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1467-8659.2004.00005.x

### Contents
1. [Install & import](#1-install--import)
2. [Synthetic data generation](#2-synthetic-data-generation)
3. [Fitting an axis-aligned ellipsoid](#3-fitting-an-axis-aligned-ellipsoid)
4. [Fitting a rotated ellipsoid](#4-fitting-a-rotated-ellipsoid)
5. [Fitting from CSV datasets](#5-fitting-from-csv-datasets)
6. [Accuracy vs noise study](#6-accuracy-vs-noise-study)
7. [Visualisation](#7-visualisation)

## 1. Install & import

In [ ]:
# If running from the repository root, the package is importable directly.
# Otherwise: pip install -e .
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

import numpy as np
import matplotlib
matplotlib.use("Agg")  # headless backend for CI / notebook export
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from rbf_ellipsoid_constraint import (
    fit_rbf_ellipsoid_linear,
    evaluate_model_linear,
    generate_ellipsoid_points,
)
print("rbf_ellipsoid_constraint imported successfully")

## 2. Synthetic data generation

The `generate_ellipsoid_points` function creates quasi-uniformly distributed points on an ellipsoid surface using the Fibonacci sphere algorithm, optionally perturbed with Gaussian noise.

In [ ]:
TRUE_CENTRE = (1.0, 2.0, 3.0)
TRUE_RADII  = (5.0, 3.0, 2.0)

pts = generate_ellipsoid_points(
    centre=TRUE_CENTRE,
    radii=TRUE_RADII,
    n_points=300,
    noise_std=0.05,
    seed=42,
)

print(f"Shape : {pts.shape}")
print(f"x range: [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}]")
print(f"y range: [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}]")
print(f"z range: [{pts[:,2].min():.2f}, {pts[:,2].max():.2f}]")

## 3. Fitting an axis-aligned ellipsoid

In [ ]:
result = fit_rbf_ellipsoid_linear(pts)
if result is not None:
    alpha, beta, centroid, scale = result
    norm_pts = (pts - centroid) / scale
    vals = evaluate_model_linear(norm_pts, norm_pts, alpha, beta)
    rms = float(np.sqrt(np.mean(vals ** 2)))
    print("=" * 55)
    print("RBF with Ellipsoid Constraint (Li et al., CGF 2004)")
    print("=" * 55)
    print(f"True  centre : {np.array(TRUE_CENTRE)}")
    print(f"Fitted centroid: {centroid.round(4)}")
    print(f"RMS implicit residual: {rms:.6f}")
    print("=" * 55)

## 4. Fitting a rotated ellipsoid

The algorithm handles arbitrary orientations without modification.

In [ ]:
from scipy.spatial.transform import Rotation

R = Rotation.from_euler('xyz', [30, 45, 60], degrees=True).as_matrix()
pts_rot = generate_ellipsoid_points(
    centre=(0.0, 0.0, 0.0),
    radii=(6.0, 4.0, 2.5),
    rotation=R,
    n_points=400,
    noise_std=0.10,
    seed=7,
)

result_rot = fit_rbf_ellipsoid_linear(pts_rot)
if result_rot is not None:
    alpha_r, beta_r, centroid_r, scale_r = result_rot
    norm_rot = (pts_rot - centroid_r) / scale_r
    vals_r = evaluate_model_linear(norm_rot, norm_rot, alpha_r, beta_r)
    rms_r = float(np.sqrt(np.mean(vals_r ** 2)))
    print(f"Centroid: {centroid_r.round(3)}")
    print(f"RMS residual: {rms_r:.4f}")

## 5. Fitting from CSV datasets

In [ ]:
data_dir = os.path.join(os.getcwd(), "..", "data")

for fname in sorted(os.listdir(data_dir)):
    if fname.endswith(".csv"):
        path = os.path.join(data_dir, fname)
        data = np.loadtxt(path, delimiter=",", skiprows=1)
        pts_csv = data[:, :3]
        r = fit_rbf_ellipsoid_linear(pts_csv)
        if r is not None:
            alpha_c, beta_c, cent_c, scale_c = r
            norm_c = (pts_csv - cent_c) / scale_c
            vals_c = evaluate_model_linear(norm_c, norm_c, alpha_c, beta_c)
            rms_c = float(np.sqrt(np.mean(vals_c ** 2)))
            print(f"{fname}")
            print(f"  centroid : {cent_c.round(3)}")
            print(f"  RMS      : {rms_c:.5f}")
            print()

## 6. Accuracy vs noise study

How does centre-estimation error grow with increasing noise?

In [ ]:
noise_levels = np.linspace(0.0, 0.5, 20)
centre_errors = []
radii_errors  = []

TRUE_C = np.array([0.0, 0.0, 0.0])
TRUE_R = np.array([4.0, 3.0, 2.0])

for sigma in noise_levels:
    p = generate_ellipsoid_points(
        centre=TRUE_C, radii=TRUE_R,
        n_points=400, noise_std=sigma, seed=0,
    )
    try:
        res = fit_rbf_ellipsoid_linear(p)
        if res is not None:
            alpha_n, beta_n, cent_n, scale_n = res
            centre_errors.append(np.linalg.norm(cent_n - TRUE_C))
            norm_n = (p - cent_n) / scale_n
            vals_n = evaluate_model_linear(norm_n, norm_n, alpha_n, beta_n)
            radii_errors.append(float(np.sqrt(np.mean(vals_n ** 2))))
        else:
            centre_errors.append(np.nan)
            radii_errors.append(np.nan)
    except ValueError:
        centre_errors.append(np.nan)
        radii_errors.append(np.nan)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(noise_levels, centre_errors, 'o-')
axes[0].set_xlabel("Noise σ"); axes[0].set_ylabel("Centre error (L2)")
axes[0].set_title("Centre recovery accuracy")
axes[1].plot(noise_levels, radii_errors, 's-', color='tab:orange')
axes[1].set_xlabel("Noise σ"); axes[1].set_ylabel("Radii error (L2)")
axes[1].set_title("Semi-axis recovery accuracy")
for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("noise_study.png", dpi=120)
plt.show()
print("Figure saved to noise_study.png")

## 7. Visualisation

Draw the fitted ellipsoid surface over the noisy data.

In [ ]:
pts_vis = generate_ellipsoid_points(
    centre=(1.0, 2.0, 3.0), radii=(5.0, 3.0, 2.0),
    n_points=300, noise_std=0.05, seed=42,
)
res_vis = fit_rbf_ellipsoid_linear(pts_vis)
if res_vis is not None:
    alpha_v, beta_v, cent_v, scale_v = res_vis
    fig = plt.figure(figsize=(9, 7))
    ax  = fig.add_subplot(111, projection='3d')
    ax.scatter(pts_vis[:,0], pts_vis[:,1], pts_vis[:,2], s=4, alpha=0.4, label='Noisy data')
    ax.set_title("RBF with Ellipsoid Constraint (Li et al., CGF 2004)")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
    ax.legend()
    plt.tight_layout()
    plt.savefig("ellipsoid_fit_vis.png", dpi=120)
    plt.show()
    print("Figure saved to ellipsoid_fit_vis.png")